In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [2]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
#preprocess data
#remove irrelevant data
data=data.drop(['RowNumber','CustomerId','Surname'], axis=1)

In [4]:
#encode categorical variable
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [6]:
#one hot encode geography
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder(drop='first')
geo_encoded=ohe.fit_transform(data[['Geography']])
geo_encoded=geo_encoded.toarray()

In [7]:
geo_df= pd.DataFrame(
  geo_encoded,
  columns=ohe.get_feature_names_out(['Geography'])

)
data=pd.concat([data, geo_df],axis=1)



In [8]:
data=data.drop(['Geography'], axis=1)

In [9]:
#divide data into independent and dependent data

X=data.drop('EstimatedSalary',axis=1)
y=data['EstimatedSalary']

In [10]:
#split train test
X_train,X_test,y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [11]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [12]:
#save encoders and scalers
with open('label_encoder_gender.pkl','wb') as file:
  pickle.dump(label_encoder_gender, file)

with open('ohe.pkl', 'wb') as file:
  pickle.dump(ohe, file)

with open('scaler.pkl', 'wb') as file:
  pickle.dump(scaler, file)


## ANN Regression


In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime


In [14]:
#build ANN model
model=Sequential([
  Dense(64, activation='relu', input_shape=(X_train.shape[1],)), #first hidden layer connected with input layer
  Dense(32, activation='relu'),#hidden layer 2
  Dense(1)#output layer
]

)

In [15]:
model.compile(optimizer='adam',loss='mean_absolute_error', metrics=['mae'])

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                768       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2881 (11.25 KB)
Trainable params: 2881 (11.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [16]:
#set up tensor board
import datetime
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir="regressionlogs/fit"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir, histogram_freq=1)



In [17]:
## set up early stopping where loss not decreasing anymore
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)
 

In [18]:
#training the model
history=model.fit(
  X_train, y_train, validation_data=(X_test, y_test), epochs=100,
  callbacks=[tensorflow_callback, early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 1s 3ms/step - loss: 100377.0781 - mae: 100377.0781 - val_loss: 98521.5781 - val_mae: 98521.5781
Epoch 2/100
250/250 [==============================] - 1s 2ms/step - loss: 99660.4766 - mae: 99660.4766 - val_loss: 97088.1875 - val_mae: 97088.1875
Epoch 3/100
250/250 [==============================] - 0s 2ms/step - loss: 97170.7422 - mae: 97170.7422 - val_loss: 93456.5938 - val_mae: 93456.5938
Epoch 4/100
250/250 [==============================] - 0s 2ms/step - loss: 92300.3359 - mae: 92300.3359 - val_loss: 87403.1016 - val_mae: 87403.1016
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 85200.9922 - mae: 85200.9922 - val_loss: 79515.9219 - val_mae: 79515.9219
Epoch 6/100
250/250 [==============================] - 0s 2ms/step - loss: 76686.2344 - mae: 76686.2344 - val_loss: 71009.2500 - val_mae: 71009.2500
Epoch 7/100
250/250 [==============================] - 0s 2ms/step - loss: 68295.0938 - mae: 68295.093

In [19]:
# load tensorboard extension
%load_ext tensorboard

In [20]:
%tensorboard --logdir regressionlogs/fit --port 6008





Reusing TensorBoard on port 6008 (pid 24964), started 0:24:44 ago. (Use '!kill 24964' to kill it.)

In [21]:
#evaluate model on the test data
test_loss, test_mae=model.evaluate(X_test, y_test)
print(f'Test MAE: {test_mae}')

63/63 [==============================] - 0s 1ms/step - loss: 50165.6914 - mae: 50165.6914
Test MAE: 50165.69140625


In [22]:
train_loss, train_mae = model.evaluate(X_train, y_train)
print("Train MAE:", train_mae)

250/250 [==============================] - 0s 1ms/step - loss: 49419.4336 - mae: 49419.4336
Train MAE: 49419.43359375


In [24]:
model.save('regression_model.h5')

d:\conda_envs\DL1\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
